### **MiniGPT**

In [10]:
import pandas as pd
import torch
import torch.nn as nn

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
df = pd.read_csv('data/Eng-Fra.csv')
df.head()

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [12]:
from preprocessing.normalize import normalize_text

df['English'] = df['English'].apply(normalize_text)
df['French'] = df['French'].apply(normalize_text)

In [13]:
df.head()

,English,French
0,go,va
1,go,marche
2,go,en route
3,go,bouge
4,hi,salut


In [40]:
sentences = []

for eng in zip(df['English']):
    sentences.append(str(eng))

print(len(sentences))

239189


In [45]:
from preprocessing.vocabulary import Vocabulary

eng_vocab = Vocabulary("eng")

# Step 1: count words
for eng in sentences:
    eng_vocab.add_sentence(eng)
# Step 2: build vocab (top-K)
eng_vocab.build_vocab(max_size=10000)

In [ ]:
from utils.GPTDataset import GPTDataset
from models.mini_gpt import MiniGPT
from torch.utils.data import DataLoader

dataset = GPTDataset(sentences, eng_vocab, seq_len=5)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniGPT(
    vocab_size=eng_vocab.vocab_size,
    d_model=128,
    heads=4,
    num_layers=2,
    max_len=50
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [44]:
num_epochs = 2

for epoch in range(num_epochs):

    total_loss = 0

    for x, y in dataloader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(x)   # [batch, seq_len, vocab]

        output = output.view(-1, output.shape[-1])
        y = y.view(-1)

        loss = criterion(output, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

KeyboardInterrupt: 